## YOLOv11n 재학습 (70:15 분할 데이터로)

 셀 1: 라이브러리 import 및 GPU 확인

In [1]:
# ============================================
# YOLOv11n 재학습 (70:15 데이터)
# ============================================

# YOLO 모델 사용을 위한 라이브러리
from ultralytics import YOLO

# 파일 경로 처리를 위한 라이브러리
from pathlib import Path

# PyTorch 라이브러리 (GPU 확인 및 딥러닝)
import torch

# 시간 측정 라이브러리
import time
# 날짜/시간 포맷팅 라이브러리
from datetime import datetime

# 헤더 출력
print("=" * 60)
print("🔥 YOLOv11n 재학습")
print("=" * 60)

# GPU 확인
# CUDA(NVIDIA GPU) 사용 가능 여부 확인
print(f"\nCUDA 사용 가능: {torch.cuda.is_available()}")

# GPU가 있으면 정보 출력
if torch.cuda.is_available():
    # GPU 이름 출력
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    # GPU 메모리 크기 출력 (GB 단위)
    print(f"메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    # GPU 없으면 CPU 모드 경고
    print("⚠️ CPU 모드로 실행됩니다 (느림)")

print("=" * 60)

🔥 YOLOv11n 재학습

CUDA 사용 가능: True
GPU: NVIDIA GeForce RTX 4060 Ti
메모리: 8.0 GB


셀 2: 경로 설정 및 확인

In [2]:
# ============================================
# 경로 설정 및 데이터 확인
# ============================================

# 프로젝트 루트 경로 설정
PROJECT_ROOT = Path(r'N:\개인\이수빈\3.13_Mini_Project')

# 학습 데이터 경로 (70:15 비율로 분할된 데이터)
DATA_DIR = PROJECT_ROOT / 'DATASET' / '5000_split_70_15'

# YOLO 학습용 설정 파일 경로 (train/val 경로, 클래스 정보 포함)
DATA_YAML = DATA_DIR / 'data_70_15.yaml'

# 학습 결과 저장 경로 (v11n용)
RESULTS_DIR = PROJECT_ROOT / 'results' / 'yolov11n_70_15'

# 경로 정보 출력
print("\n[경로 확인]")
print(f"데이터: {DATA_DIR}")
print(f"YAML:  {DATA_YAML}")
print(f"결과:  {RESULTS_DIR}")

# YAML 파일 존재 여부 확인
if DATA_YAML.exists():
    # 파일이 있으면 OK
    print(f"\n✅ data.yaml 존재")
else:
    # 파일이 없으면 에러 (학습 불가)
    print(f"\n❌ data.yaml 없음!")
    print(f"   경로 확인: {DATA_YAML}")

# Train 폴더 존재 및 이미지 개수 확인
train_dir = DATA_DIR / 'train' / 'images'
if train_dir.exists():
    # Train 이미지 개수 세기
    train_count = len(list(train_dir.glob('*.jpg')))
    print(f"✅ Train: {train_count:,}장")
else:
    # Train 폴더 없으면 에러
    print(f"❌ Train 폴더 없음")

# Validation 폴더 존재 및 이미지 개수 확인
val_dir = DATA_DIR / 'val' / 'images'
if val_dir.exists():
    # Val 이미지 개수 세기
    val_count = len(list(val_dir.glob('*.jpg')))
    print(f"✅ Val:   {val_count:,}장")
else:
    # Val 폴더 없으면 에러
    print(f"❌ Val 폴더 없음")

print("=" * 60)


[경로 확인]
데이터: N:\개인\이수빈\3.13_Mini_Project\DATASET\5000_split_70_15
YAML:  N:\개인\이수빈\3.13_Mini_Project\DATASET\5000_split_70_15\data_70_15.yaml
결과:  N:\개인\이수빈\3.13_Mini_Project\results\yolov11n_70_15

✅ data.yaml 존재
✅ Train: 4,172장
✅ Val:   894장


셀 3: 모델 로드

In [3]:
# ============================================
# YOLOv11n 모델 로드
# ============================================

print("\n[모델 로드]")

# YOLOv11n 사전 학습된 가중치 파일 로드
# 'yolo11n.pt'는 Ultralytics에서 제공하는 사전학습 모델
# 처음 실행 시 자동으로 다운로드됨
model = YOLO('yolo11n.pt')

print("✅ YOLOv11n 사전학습 모델 로드 완료")
print("=" * 60)


[모델 로드]
✅ YOLOv11n 사전학습 모델 로드 완료


셀 4: 학습 시작

In [4]:
# ============================================
# 학습 시작
# ============================================

print("\n[학습 시작]")
# 현재 시각 출력 (시작 시간 기록)
print(f"시작 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

# 시작 시간 기록 (소요 시간 계산용)
start_time = time.time()

# YOLO 모델 학습 실행
results = model.train(
    data=str(DATA_YAML),        # 데이터 설정 파일 경로 (train/val 경로, 클래스 정보)
    epochs=100,                  # 학습 반복 횟수 (100 에폭)
    imgsz=640,                   # 입력 이미지 크기 (640x640으로 리사이즈)
    batch=16,                    # 배치 크기 (한 번에 16장씩 처리) - 메모리 부족 시 8로 변경
    device=0,                    # 사용할 디바이스 (0=첫번째 GPU, 'cpu'=CPU)
    project=str(RESULTS_DIR.parent),  # 결과 저장 상위 폴더
    name=RESULTS_DIR.name,       # 결과 저장 폴더명
    patience=50,                 # Early stopping 인내 (50 에폭 동안 개선 없으면 중단)
    save=True,                   # 체크포인트 저장 여부 (best.pt, last.pt 저장)
    plots=True,                  # 학습 그래프 생성 여부 (confusion matrix, results.png 등)
    val=True,                    # Validation 실행 여부 (에폭마다 검증)
    verbose=True                 # 상세 출력 여부 (진행 상황 자세히 표시)
)

# 종료 시간 기록
end_time = time.time()

# 소요 시간 계산 (초 단위)
elapsed = end_time - start_time

# 완료 메시지 출력
print("=" * 60)
print(f"✅ 학습 완료!")
# 종료 시각 출력
print(f"종료 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
# 소요 시간 출력 (분, 시간 단위)
print(f"소요 시간: {elapsed/60:.1f}분 ({elapsed/3600:.2f}시간)")
print("=" * 60)


[학습 시작]
시작 시각: 2026-03-03 10:41:51
Ultralytics 8.4.19  Python-3.10.19 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=N:\\\3.13_Mini_Project\DATASET\5000_split_70_15\data_70_15.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov11n_70_15, nbs=64, nms=Fal

셀 5: 최종 결과 확인

In [5]:
# ============================================
# 최종 결과 파일 확인
# ============================================

print("\n[최종 결과]")
print("=" * 60)

# Best 모델 파일 경로 (가장 성능 좋은 모델)
best_model = RESULTS_DIR / 'weights' / 'best.pt'

# Best 모델 파일 존재 확인
if best_model.exists():
    # 파일이 있으면 경로 출력
    print(f"✅ Best 모델: {best_model}")
    
    # 모델 파일 크기 확인 (MB 단위)
    size_mb = best_model.stat().st_size / 1024**2
    print(f"   크기: {size_mb:.2f} MB")
else:
    # 파일이 없으면 경고
    print(f"⚠️ Best 모델 없음")

# 학습 결과 CSV 파일 확인 (에폭별 성능 기록)
results_csv = RESULTS_DIR / 'results.csv'
if results_csv.exists():
    print(f"✅ 결과 CSV: {results_csv}")

# Confusion Matrix 이미지 확인 (예측 vs 실제 비교 표)
confusion_matrix = RESULTS_DIR / 'confusion_matrix.png'
if confusion_matrix.exists():
    print(f"✅ Confusion Matrix: {confusion_matrix}")

# 학습 그래프 이미지 확인 (loss, mAP 등)
results_png = RESULTS_DIR / 'results.png'
if results_png.exists():
    print(f"✅ 학습 그래프: {results_png}")

# 전체 결과 폴더 위치 출력
print(f"\n📂 전체 결과 위치:")
print(f"   {RESULTS_DIR}")

print("=" * 60)
print("🎉 YOLOv11n 학습 완료!")
print("=" * 60)


[최종 결과]
✅ Best 모델: N:\개인\이수빈\3.13_Mini_Project\results\yolov11n_70_15\weights\best.pt
   크기: 5.23 MB
✅ 결과 CSV: N:\개인\이수빈\3.13_Mini_Project\results\yolov11n_70_15\results.csv
✅ Confusion Matrix: N:\개인\이수빈\3.13_Mini_Project\results\yolov11n_70_15\confusion_matrix.png
✅ 학습 그래프: N:\개인\이수빈\3.13_Mini_Project\results\yolov11n_70_15\results.png

📂 전체 결과 위치:
   N:\개인\이수빈\3.13_Mini_Project\results\yolov11n_70_15
🎉 YOLOv11n 학습 완료!


셀 6: Validation 성능 확인

In [6]:
# ============================================
# Validation 성능 확인
# ============================================

print("\n[Validation 성능]")
print("=" * 60)

# Best 모델 파일 로드
best_model_path = RESULTS_DIR / 'weights' / 'best.pt'
best_model = YOLO(str(best_model_path))

# Validation 데이터로 성능 평가 실행
val_results = best_model.val(
    data=str(DATA_YAML),   # 데이터 설정 파일 (val 경로 포함)
    imgsz=640,              # 이미지 크기
    batch=16,               # 배치 크기
    device=0                # GPU 0번 사용
)

# 주요 성능 지표 출력
print(f"\n📊 주요 지표:")
# mAP50: IoU 50%에서의 평균 정밀도
print(f"  mAP50:     {val_results.box.map50:.3f}")
# mAP50-95: IoU 50-95%에서의 평균 정밀도
print(f"  mAP50-95:  {val_results.box.map:.3f}")
# Precision: 정밀도 (예측한 것 중 맞은 비율)
print(f"  Precision: {val_results.box.mp:.3f}")
# Recall: 재현율 (실제 중 찾은 비율)
print(f"  Recall:    {val_results.box.mr:.3f}")

print("=" * 60)


[Validation 성능]
Ultralytics 8.4.19  Python-3.10.19 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8188MiB)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.20.0 ms, read: 49.68.9 MB/s, size: 209.4 KB)
val: Scanning N:\개인\이수빈\3.13_Mini_Project\DATASET\5000_split_70_15\val\labels.cache... 893 images, 430 backgrounds, 2 corrupt: 100% ━━━━━━━━━━━━ 894/894  0.0s
val: N:\\\3.13_Mini_Project\DATASET\5000_split_70_15\val\images\fire_0230.jpg: ignoring corrupt image/label: cannot identify image file 'N:\\\\\\3.13_Mini_Project\\DATASET\\5000_split_70_15\\val\\images\\fire_0230.jpg'
val: N:\\\3.13_Mini_Project\DATASET\5000_split_70_15\val\images\fire_0509.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.08692]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 56/56 15.4it/s 3.6s0.1s
                   all        892       1154      0